# [Open in Colab](https://colab.research.google.com/github/dmochow/BME-i9400-2026/blob/main/notebooks/meeting13_optimization_generalization.ipynb)

# BME i9400 — Meeting 13
## Optimisation and Generalisation

**Fall 2026 · Monday, November 2**

Today's lecture is about how a model's weights are found when there is no formula for them. It
develops gradient descent, analyses the learning rate and the shape of the loss surface, introduces
stochastic gradient descent, and shows that the number of steps taken is itself a form of
regularisation.

**Agenda**

1. Fitting is optimisation
2. Gradient descent
3. The learning rate
4. Conditioning and the loss surface
5. Stochastic gradient descent
6. Convergence
7. Optimisation and generalisation
8. Connection to the course

---
## 1 — Fitting is optimisation

Every model in the course so far was fitted by minimising a loss averaged over the training samples:

$$L(w) = \frac{1}{n}\sum_{i=1}^{n} \ell_i(w)$$

> $\ell_i(w)$ is the loss on sample $i$: the squared error $(y_i - x_i^{\top} w)^2$ for linear
> regression (Meeting 6), the cross-entropy for logistic regression (Meeting 8), with a penalty added
> for ridge and the lasso (Meetings 7 and 11). Dividing by $n$ does not move the minimum; it keeps the
> loss on the same scale whatever the size of the training set.

Only least squares and ridge have a formula for the minimum, the normal equations. Logistic regression
has none, the lasso has none, and the neural networks of Meeting 17 have none. For all of these the
weights are found **iteratively**: start somewhere, and repeatedly move to a point with lower loss.
Meeting 8, section 4 did this for one feature. Today asks how to choose the step, how fast the
iteration converges, and what happens to generalisation along the way.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
plt.rcParams.update({"font.size": 12, "figure.dpi": 110})

# The diabetes-progression data of Meetings 6, 7 and 11. Two features for the pictures: blood pressure and s5.
prog  = load_diabetes(scaled=False, as_frame=True).frame
X2    = prog[["bp", "s5"]].values
y2    = prog["target"].values - prog["target"].values.mean()     # centred, so no intercept is needed
X2c   = X2 - X2.mean(axis=0)                                      # centred, in original units
X2s   = X2c / X2.std(axis=0)                                      # standardised

def mse(w, X, y):
    return np.mean((y - X @ w) ** 2)

def mse_grad(w, X, y):
    return (2 / len(y)) * X.T @ (X @ w - y)

def gradient_descent(grad, w0, eta, steps):
    # Repeat w <- w - eta * grad(w); return every iterate.
    path = [np.array(w0, dtype=float)]
    for _ in range(steps):
        path.append(path[-1] - eta * grad(path[-1]))
    return np.array(path)

print(f"{len(y2)} patients; blood pressure sd {X2[:, 0].std():.1f} mm Hg, s5 sd {X2[:, 1].std():.2f} (log units)")

---
## 2 — Gradient descent

Near a point $w$, the loss is approximated by its first-order Taylor expansion:

$$L(w + \Delta) \approx L(w) + \nabla L(w)^{\top} \Delta$$

> The **gradient** $\nabla L(w)$ is the vector of partial derivatives $\partial L / \partial w_j$. It
> points in the direction in which $L$ increases fastest, and its length is the rate of increase in
> that direction.

Choose the step to point against the gradient, $\Delta = -\eta\, \nabla L(w)$ with $\eta > 0$:

$$L\big(w - \eta \nabla L(w)\big) \approx L(w) - \eta\, \nabla L(w)^{\top} \nabla L(w) = L(w) - \eta\, \|\nabla L(w)\|^2$$

The change is negative whenever the gradient is not zero, so a small enough step always lowers the
loss. Repeating it gives **gradient descent**:

$$\boxed{\; w_{t+1} = w_t - \eta\, \nabla L(w_t) \;}$$

> $\eta$ (eta) is the **learning rate**, or step size. It is a hyperparameter of the fitting
> procedure, not of the model. The approximation above holds only for small steps, which is why
> $\eta$ cannot be made arbitrarily large (section 3).

For the squared error the gradient was derived in Meeting 6:

$$\nabla L(w) = \frac{2}{n} X^{\top}\left(Xw - y\right)$$

The cell below runs gradient descent from $w = 0$ on two standardised features, blood pressure and
`s5`, and draws the path over the **contours** of the loss.

> A **contour** is a curve along which the loss is constant, like a line of equal height on a
> topographic map. The gradient at each point is perpendicular to the contour through it, so
> gradient descent crosses the contours at right angles.

In [ ]:
w_star = np.linalg.solve(X2s.T @ X2s, X2s.T @ y2)                 # the exact minimum, for reference
path   = gradient_descent(lambda w: mse_grad(w, X2s, y2), w0=[0, 0], eta=0.1, steps=30)

for t in [0, 1, 2, 5, 10, 30]:
    print(f"step {t:2d}:  w = ({path[t, 0]:6.2f}, {path[t, 1]:6.2f})   loss {mse(path[t], X2s, y2):8.1f}")
print(f"minimum:  w = ({w_star[0]:6.2f}, {w_star[1]:6.2f})   loss {mse(w_star, X2s, y2):8.1f}")

In [ ]:
def contour(ax, X, y, w_star, span, levels=14):
    a = np.linspace(w_star[0] - span[0], w_star[0] + span[0], 200)
    b = np.linspace(w_star[1] - span[1], w_star[1] + span[1], 200)
    A, B = np.meshgrid(a, b)
    W = np.stack([A.ravel(), B.ravel()], axis=1)
    Lv = np.mean((y[None, :] - W @ X.T) ** 2, axis=1).reshape(A.shape)
    ax.contour(A, B, Lv, levels=np.quantile(Lv, np.linspace(0.005, 0.9, levels)) , colors="#b8b3c2", linewidths=1)
    ax.plot(*w_star, "*", ms=14, color="#1a1a1a")

fig, ax = plt.subplots(figsize=(6.4, 5.6))
contour(ax, X2s, y2, w_star, span=(40, 40))
ax.plot(path[:, 0], path[:, 1], "o-", ms=4, color="#5b3a8e", lw=1.8, label="gradient descent, $\\eta$ = 0.1")
ax.set_xlabel("weight on blood pressure (standardised)"); ax.set_ylabel("weight on s5 (standardised)")
ax.set_title("Gradient descent on the squared error"); ax.set_aspect("equal"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

---
## 3 — The learning rate

For the squared error the behaviour of gradient descent can be worked out exactly. The loss is a
quadratic, with second-derivative matrix

$$H = \frac{2}{n} X^{\top} X$$

> $H$ is the **Hessian**: the matrix of second partial derivatives $\partial^2 L / \partial w_j \partial w_k$.
> It describes the curvature of the loss. For the squared error it is the same at every $w$.

Let $e_t = w_t - w^*$ be the distance from the minimum. The gradient is zero at $w^*$, so
$\nabla L(w_t) = \nabla L(w_t) - \nabla L(w^*) = H (w_t - w^*) = H e_t$, and one step gives

$$e_{t+1} = w_{t+1} - w^* = w_t - \eta H e_t - w^* = e_t - \eta H e_t = (I - \eta H)\, e_t$$

After $t$ steps, $e_t = (I - \eta H)^t e_0$.

### One direction at a time

Write $e_0$ in the eigenvectors of $H$ (Meeting 4), with eigenvalues $h_1 \ge h_2 \ge \dots \ge h_d > 0$.
Along eigenvector $k$, each step multiplies the error by the same number:

$$e_{t}^{(k)} = \left(1 - \eta\, h_k\right)^{t}\, e_0^{(k)}$$

> $h_k$ is the curvature of the loss along eigenvector $k$: a large $h_k$ is a steep, narrow valley,
> a small $h_k$ a shallow, wide one.

The error shrinks along every direction if and only if $|1 - \eta h_k| < 1$ for every $k$, that is

$$\boxed{\; 0 < \eta < \frac{2}{h_{\max}} \;}$$

- $\eta$ small: every factor is just below $1$; convergence is slow, most of all along the flattest
  direction, whose factor is $1 - \eta h_{\min}$
- $\eta$ between $1/h_{\max}$ and $2/h_{\max}$: the factor along the steepest direction is negative,
  and the iterates **oscillate** across the valley while still converging
- $\eta > 2/h_{\max}$: that factor exceeds $1$ in magnitude, and the error **grows** at every step

In [ ]:
H_s     = (2 / len(y2)) * X2s.T @ X2s
h       = np.linalg.eigvalsh(H_s)                                  # ascending
eta_max = 2 / h[-1]
print(f"eigenvalues of H: {h[0]:.2f} and {h[-1]:.2f}     largest stable learning rate 2 / h_max = {eta_max:.3f}\n")

runs = {}
for frac in [0.1, 0.5, 0.95, 1.05]:
    eta  = frac * eta_max
    p    = gradient_descent(lambda w: mse_grad(w, X2s, y2), [0, 0], eta, 40)
    runs[frac] = (eta, p)
    gap  = np.array([mse(w, X2s, y2) for w in p]) - mse(w_star, X2s, y2)
    print(f"eta = {eta:.3f} ({frac:4.2f} x 2/h_max):  loss above the minimum after 5 steps {gap[5]:10.3g},  after 40 steps {gap[40]:10.3g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
contour(axes[0], X2s, y2, w_star, span=(40, 40))
for frac, col in [(0.1, "#2b6cb0"), (0.5, "#2f8f5b"), (0.95, "#c1272d")]:
    eta, p = runs[frac]
    axes[0].plot(p[:25, 0], p[:25, 1], "o-", ms=3, lw=1.4, color=col, label=f"$\\eta$ = {frac} x 2/$h_{{max}}$")
axes[0].set_aspect("equal"); axes[0].legend(loc="lower right", fontsize=10); axes[0].set_title("Paths, first 25 steps")
axes[0].set_xlabel("weight on blood pressure"); axes[0].set_ylabel("weight on s5")
for frac, col in [(0.1, "#2b6cb0"), (0.5, "#2f8f5b"), (0.95, "#c1272d"), (1.05, "#1a1a1a")]:
    eta, p = runs[frac]
    gap = np.array([mse(w, X2s, y2) for w in p]) - mse(w_star, X2s, y2)
    axes[1].semilogy(gap, lw=2, color=col, label=f"{frac} x 2/$h_{{max}}$")
axes[1].set_xlabel("step"); axes[1].set_ylabel("loss above the minimum"); axes[1].set_title("Convergence")
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

The four runs follow the analysis. At $0.1 \times 2/h_{\max}$ the loss falls steadily but slowly. At
$0.5 \times 2/h_{\max} = 1/h_{\max}$ the steepest direction is removed in a single step and the
remaining error decays quickly. At $0.95 \times 2/h_{\max}$ the path zig-zags across the valley; the
steep direction is barely damped, and convergence is slower than at half the rate. At $1.05 \times
2/h_{\max}$ the loss grows by orders of magnitude within 40 steps.

> For a loss that is not quadratic, such as the cross-entropy, $H$ changes from point to point and the
> bound holds only locally, but the three regimes — slow, oscillating, diverging — are the same. A
> loss that rises or turns into `nan` in the first few steps almost always means that $\eta$ is too
> large (Meeting 18).

---
## 4 — Conditioning and the loss surface

Section 3 gives the best achievable rate. The learning rate is limited by the steepest direction,
$\eta < 2/h_{\max}$, while the error along the flattest direction shrinks by $1 - \eta h_{\min}$ per
step. Even at $\eta = 1/h_{\max}$ that factor is $1 - h_{\min}/h_{\max}$. The ratio

$$\kappa = \frac{h_{\max}}{h_{\min}}$$

is the **condition number** of the problem. When $\kappa$ is large, the loss surface is a long,
narrow valley: steep across, nearly flat along. A learning rate safe for the steep direction makes
almost no progress along the flat one, and the number of steps needed grows in proportion to $\kappa$.

> Two things make $\kappa$ large in practice: features on very different **scales**, and features that
> are strongly **correlated** (the `s1`–`s2` pair of Meeting 7, whose correlation matrix had an
> eigenvalue below one percent of the others).

Blood pressure is measured in mm Hg, with a standard deviation of about 14; `s5` is a logarithm, with
a standard deviation of about 0.5. The cell below fits the same model to the centred features in
their original units and to the standardised features, each at $\eta = 1/h_{\max}$.

In [ ]:
w_raw = np.linalg.solve(X2c.T @ X2c, X2c.T @ y2)
results = {}
for name, Xf, wf in [("original units", X2c, w_raw), ("standardised", X2s, w_star)]:
    hf = np.linalg.eigvalsh((2 / len(y2)) * Xf.T @ Xf)
    p  = gradient_descent(lambda w: mse_grad(w, Xf, y2), [0, 0], 1 / hf[-1], 2000)
    gap = np.array([mse(w, Xf, y2) for w in p]) - mse(wf, Xf, y2)
    steps_1pct = np.argmax(gap < 0.01 * gap[0])
    results[name] = (Xf, wf, p)
    print(f"{name:<15} condition number {hf[-1] / hf[0]:6.1f}   steps to remove 99% of the excess loss: {steps_1pct}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
for ax, (name, (Xf, wf, p)), span in zip(axes, results.items(), [(4, 250), (40, 40)]):
    contour(ax, Xf, y2, wf, span=span)
    ax.plot(p[:200, 0], p[:200, 1], ".-", ms=3, lw=1, color="#c1272d", label="first 200 steps")
    ax.set_title(name); ax.set_xlabel("weight on blood pressure"); ax.set_ylabel("weight on s5"); ax.legend(loc="lower right")
axes[1].set_aspect("equal")
plt.tight_layout(); plt.show()

In original units the condition number is about 830, and gradient descent needs more than 1,600 steps
to remove 99% of the excess loss: in the first steps it moves almost only the blood-pressure weight,
whose direction is steep, and it then creeps along the floor of the valley toward the minimum. (The two
axes of the left panel differ in scale by a factor of about 60, which is why the valley does not look
narrow.) After standardisation the condition number is about 2, set only by the correlation of $0.39$
between the two features, and two steps suffice. This is why Meeting 8 standardised
glucose before running gradient descent, and why every pipeline since has a `StandardScaler` before
the model.

> For two standardised features with correlation $r$, the eigenvalues of the correlation matrix are
> $1 + r$ and $1 - r$, so $\kappa = (1 + r)/(1 - r)$. Standardisation removes the effect of scale; it
> cannot remove the effect of correlation, which grows without bound as $r \to 1$.

---
## 5 — Stochastic gradient descent

Each step of gradient descent computes the gradient on every training sample:

$$\nabla L(w) = \frac{1}{n}\sum_{i=1}^{n} \nabla \ell_i(w)$$

With millions of samples — every pixel of every image, every beat of every ECG — one exact gradient is
expensive, and most of its precision is wasted far from the minimum. **Stochastic gradient descent**
(SGD) replaces it with the average over a small random subset $B$ of the samples, a **minibatch**:

$$g_B(w) = \frac{1}{|B|}\sum_{i \in B} \nabla \ell_i(w), \qquad w_{t+1} = w_t - \eta\, g_B(w_t)$$

If $B$ is drawn uniformly at random, every sample is equally likely to contribute, so

$$\mathbb{E}\big[g_B(w)\big] = \frac{1}{n}\sum_{i=1}^{n} \nabla \ell_i(w) = \nabla L(w)$$

> The minibatch gradient is an **unbiased** estimate of the full gradient: correct on average, noisy
> in any single step. Its variance falls in proportion to $1/|B|$.
>
> An **epoch** is one pass through the training set. With $n$ samples and batch size $|B|$, an epoch
> of SGD takes $n/|B|$ steps for the same computational cost as one step of full gradient descent.

The cell below fits the logistic regression of Meetings 8 to 12 — all eight measurements, imputed and
standardised — four ways, for 30 epochs each, and records how far the cross-entropy is above its
minimum after every epoch:

- full-batch gradient descent, $\eta = 2$
- minibatches of 32, $\eta = 0.5$
- single samples ($|B| = 1$), constant $\eta = 0.05$
- single samples, with a **decaying learning rate** $\eta_e = 0.5/(1 + e)$ in epoch $e$

> The gradient of the cross-entropy is $\nabla \ell_i = (p_i - y_i)\, x_i$ (Meeting 8, section 4).
> The imputation and standardisation here use all 768 women, because the question is how fast the
> training loss is minimised, not how well the model generalises.

In [ ]:
from sklearn.linear_model import LogisticRegression

pima = pd.read_csv("https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/diabetes.csv")
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
pima[cols] = pima[cols].replace(0, np.nan)
Xp = pima.drop(columns="Outcome"); Xp = Xp.fillna(Xp.median()).values
Zp = (Xp - Xp.mean(axis=0)) / Xp.std(axis=0)
Xd, yp = np.column_stack([np.ones(len(Zp)), Zp]), pima["Outcome"].values

sigmoid = lambda z: 1 / (1 + np.exp(-z))
def xent(w):
    p = sigmoid(Xd @ w)
    return -np.mean(yp * np.log(p) + (1 - yp) * np.log(1 - p))

exact  = LogisticRegression(penalty=None, max_iter=10_000, tol=1e-12).fit(Zp, yp)
L_min  = xent(np.r_[exact.intercept_, exact.coef_[0]])
print(f"minimum cross-entropy (scikit-learn): {L_min:.5f}")

In [ ]:
def sgd(batch, eta, epochs=30, decay=False, seed=0):
    # Minibatch SGD on the cross-entropy; returns the loss above the minimum after each epoch.
    rng, w, gaps = np.random.default_rng(seed), np.zeros(Xd.shape[1]), []
    for e in range(epochs):
        order = rng.permutation(len(yp))                            # a fresh shuffle every epoch
        lr = eta / (1 + e) if decay else eta
        for start in range(0, len(yp), batch):
            B = order[start:start + batch]
            w = w - lr * Xd[B].T @ (sigmoid(Xd[B] @ w) - yp[B]) / len(B)
        gaps.append(xent(w) - L_min)
    return np.array(gaps)

runs_sgd = {"full batch (gradient descent), eta = 2":  sgd(len(yp), 2.0),
            "batches of 32, eta = 0.5":                 sgd(32, 0.5),
            "single samples, eta = 0.05":               sgd(1, 0.05),
            "single samples, eta = 0.5 / (1 + epoch)":  sgd(1, 0.5, decay=True)}
for name, g in runs_sgd.items():
    print(f"{name:<42} above the minimum after 1 epoch {g[0]:8.4f}   after 30 epochs {g[-1]:8.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
for (name, g), col in zip(runs_sgd.items(), ["#1a1a1a", "#5b3a8e", "#c1272d", "#2f8f5b"]):
    ax.semilogy(np.arange(1, len(g) + 1), g, "o-", ms=3, lw=2, color=col, label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy above its minimum")
ax.set_title("Logistic regression, 768 women: one epoch costs the same in every run")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

For the same computation, minibatches of 32 are six times closer to the minimum than full-batch
gradient descent after one epoch, because they take 24 steps where gradient descent takes one. They
then stop improving: each step is noisy, and with a constant learning rate the iterates keep moving
around the minimum at a distance set by $\eta$ and the batch size, instead of settling on it. Single
samples with a constant rate show the same floor, higher and more erratic. Decaying the learning rate
shrinks the steps as the iterates approach the minimum, and the loss falls with fluctuations.

Full-batch gradient descent passes every SGD run after about 12 epochs and keeps descending steadily.
With 768 samples an exact gradient is cheap, and it wins. SGD is preferred when $n$ is so large that
a few epochs are all that can be afforded, and a loss within $10^{-3}$ of its minimum is far more
precise than the noise in the data can justify.

> SGD with a constant learning rate converges to a **neighbourhood** of the minimum, not to the
> minimum itself. Reaching the minimum requires a **learning-rate schedule** that decreases the rate
> over time. In practice the schedule is often a few constant stages, each smaller than the last.

---
## 6 — Convergence

**Convex and non-convex losses.** The squared error, the cross-entropy of logistic regression, and
their ridge and lasso versions are **convex**: the loss surface is a single bowl, and every local
minimum is the global minimum. Gradient descent with a suitable learning rate reaches it from any
starting point.

> A function is **convex** if the straight line between any two points on its graph lies on or above
> the graph. For a twice-differentiable function, this is equivalent to the Hessian having no negative
> eigenvalues anywhere.

The loss of a neural network (Meeting 17) is not convex. It can have many local minima and many
**saddle points**, where the gradient is zero but the point is a minimum along some directions and
a maximum along others. The result of gradient descent then depends on where it starts — which is why
neural networks are trained several times from different random starting weights.

In [ ]:
f  = lambda w: 0.05 * w**4 - 0.6 * w**2 + 0.35 * w + 2
df = lambda w: 0.2 * w**3 - 1.2 * w + 0.35
grid = np.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(grid, f(grid), lw=2.5, color="#6e6878")
for w0, col in [(-3.5, "#2b6cb0"), (0.2, "#2f8f5b"), (3.6, "#c1272d")]:
    p = gradient_descent(df, [w0], 0.15, 60)[:, 0]
    ax.plot(p, f(p), "o-", ms=4, lw=1, color=col, label=f"start at {w0:+.1f}  ->  ends at {p[-1]:+.2f},  loss {f(p[-1]):.2f}")
ax.set_xlabel("$w$"); ax.set_ylabel("$L(w)$"); ax.set_title("A non-convex loss: the answer depends on the start")
ax.legend(fontsize=10, loc="upper center"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**When to stop.** An iterative fit stops when one of these is met:

- the gradient is small, $\|\nabla L(w_t)\| < \text{tol}$
- the loss, or the weights, change by less than a tolerance between steps
- a maximum number of steps or epochs is reached

> scikit-learn exposes these as `tol` and `max_iter`. When `max_iter` is reached first it issues a
> `ConvergenceWarning`: the weights returned are not the minimum. The warning is not a formality. The
> usual causes, in order: unstandardised features (section 4), too small a `max_iter`, and a learning
> rate or penalty that makes the problem hard.

**Momentum and adaptive methods.** Two modifications of SGD are used to train almost every neural
network. **Momentum** accumulates a running average of past gradients, which damps the zig-zag across a
narrow valley and accelerates progress along it. **Adam** also rescales each weight's step by a running
estimate of the size of its gradients, which partly compensates for poor conditioning. Both are
available in PyTorch, which the course uses from Meeting 18.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    raw_fit = LogisticRegression(penalty=None, max_iter=100).fit(Xp, yp)          # original units
    std_fit = LogisticRegression(penalty=None, max_iter=100).fit(Zp, yp)          # standardised
print(f"iterations used:  original units {raw_fit.n_iter_[0]},  standardised {std_fit.n_iter_[0]}")
print(f"convergence warnings: {sum(issubclass(w.category, ConvergenceWarning) for w in caught)}")
p_raw = raw_fit.predict_proba(Xp)[:, 1]
xent_raw = -np.mean(yp * np.log(p_raw) + (1 - yp) * np.log(1 - p_raw))
print(f"cross-entropy above the minimum:  original units {xent_raw - L_min:.5f},  "
      f"standardised {xent(np.r_[std_fit.intercept_, std_fit.coef_[0]]) - L_min:.5f}")

---
## 7 — Optimisation and generalisation

Minimising the training loss and generalising to new patients are different goals. The experiment
below makes the difference visible:

- 40 training patients from the diabetes-progression data, and the other 402 as a validation set
- the 10 real features plus 30 features of pure noise, all standardised on the training patients
- gradient descent on the squared error from $w = 0$, recording the training and validation RMSE at
  every step

With 40 features and 40 patients, least squares can fit the training set exactly, so the minimum of
the training loss is an interpolation of the noise.

In [ ]:
rng   = np.random.default_rng(0)
F     = np.hstack([prog.drop(columns="target").values, rng.normal(size=(len(prog), 30))])
y_all = prog["target"].values
order = rng.permutation(len(prog)); tr, va = order[:40], order[40:]
mu, sd = F[tr].mean(axis=0), F[tr].std(axis=0)
Ztr, Zva = (F[tr] - mu) / sd, (F[va] - mu) / sd
ybar  = y_all[tr].mean()

L_tr  = np.linalg.eigvalsh((2 / 40) * Ztr.T @ Ztr)[-1]
eta   = 0.2 / L_tr
w, steps = np.zeros(F.shape[1]), 20_000
train_rmse, val_rmse, wnorm = [], [], []
for t in range(steps):
    w = w - eta * mse_grad(w, Ztr, y_all[tr] - ybar)
    train_rmse.append(np.sqrt(np.mean((Ztr @ w + ybar - y_all[tr]) ** 2)))
    val_rmse.append(np.sqrt(np.mean((Zva @ w + ybar - y_all[va]) ** 2)))
    wnorm.append(np.linalg.norm(w))
t_best = int(np.argmin(val_rmse))
print(f"best validation RMSE {val_rmse[t_best]:.1f} at step {t_best + 1}  (training RMSE {train_rmse[t_best]:.1f})")
print(f"after {steps} steps: training RMSE {train_rmse[-1]:.2f}, validation RMSE {val_rmse[-1]:.1f}")

In [ ]:
from sklearn.linear_model import Ridge
lams = np.logspace(-1, 4, 40)
ridge_val = [np.sqrt(np.mean((Ridge(alpha=l).fit(Ztr, y_all[tr]).predict(Zva) - y_all[va]) ** 2)) for l in lams]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
it = np.arange(1, steps + 1)
axes[0].semilogx(it, train_rmse, lw=2, color="#2b6cb0", label="training")
axes[0].semilogx(it, val_rmse,   lw=2, color="#c1272d", label="validation")
axes[0].axvline(t_best + 1, ls="--", color="#c1272d", lw=1)
axes[0].set_xlabel("gradient descent step"); axes[0].set_ylabel("RMSE"); axes[0].set_ylim(0, 300)
axes[0].set_title("Error along the gradient descent path"); axes[0].legend()
axes[1].semilogx(lams, ridge_val, lw=2, color="#5b3a8e")
axes[1].axhline(val_rmse[t_best], ls="--", color="#c1272d", lw=1, label="best early-stopped gradient descent")
axes[1].invert_xaxis(); axes[1].set_ylim(0, 300)
axes[1].set_xlabel("ridge $\\lambda$  (decreasing to the right)"); axes[1].set_ylabel("validation RMSE")
axes[1].set_title("Ridge regression on the same data"); axes[1].legend()
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The training error falls at every step and reaches nearly zero: gradient descent is doing its job.
The validation error falls for the first dozen steps, while the weights pick up the real features,
and then rises far above its starting value as the weights grow to fit the noise in 40 patients. The
best validation error along the path is close to the best that ridge regression achieves on the same
data, and the two curves have the same shape when the ridge axis is reversed.

This is not a coincidence. Started from $w = 0$, gradient descent moves first along the directions of
largest curvature — the well-determined directions of section 3 — and only later along the flat,
poorly determined ones, which is what ridge suppresses. Stopping after $t$ steps behaves approximately
like ridge with $\lambda \propto 1/(\eta t)$.

> **Early stopping** is the practice of monitoring the validation loss during training and keeping
> the weights from the step where it was lowest. It is a form of regularisation, with the number of
> steps as the hyperparameter, and the standard practice for neural networks. As with every
> hyperparameter, the step chosen on the validation set makes that set's error optimistic
> (Meeting 12).

---
## 8 — Connection to the course

**Meetings 6 to 12, completed.** Every model so far was convex and was fitted to near its minimum by
scikit-learn. Today showed what that fitting does: gradient descent, limited by the steepest
direction of the loss and slowed by the flattest, which standardisation largely repairs.

**From Meeting 17 on.** Neural networks are fitted only by SGD and its variants, on non-convex losses,
with early stopping as a standard regulariser. The learning rate, the batch size, the schedule, and
the number of epochs become the hyperparameters that matter most. The Meeting 18 studio diagnoses
training runs with the symptoms of today: a loss that diverges (section 3), one that stalls
(section 4), and a validation error that turns upward (section 7).

## What to take away

- Without a formula for the minimum, weights are found by repeated steps against the gradient,
  $w_{t+1} = w_t - \eta \nabla L(w_t)$
- For a quadratic loss, gradient descent converges if and only if $\eta < 2/h_{\max}$; oscillation
  sets in above $1/h_{\max}$ and divergence above $2/h_{\max}$
- The condition number $h_{\max}/h_{\min}$ sets the number of steps; features on different scales make
  it large, and standardisation makes it small
- SGD uses minibatch gradients, which are unbiased but noisy; with a constant rate it reaches a
  neighbourhood of the minimum, and a decaying rate is needed to converge
- Convex losses have one minimum; non-convex losses depend on the start
- A `ConvergenceWarning` means the weights are not the minimum
- Training loss and validation loss diverge along the optimisation path; early stopping is
  regularisation, close to ridge

### Wednesday, November 4 — Neural networks

The multilayer perceptron: hidden layers and activations, the forward and backward passes, capacity,
and regularisation — a model trained by the methods of today.

## Recommended resources

Everything here is optional and freely available. Each item covers material from this lecture in a
different style or voice; none of it is assigned.

### Start here

| Resource | Why this one | Time |
|---|---|---|
| **[Gradient Descent, Step-by-Step](https://www.youtube.com/watch?v=sDv4f4s2SB8)** — StatQuest | Section 2 for a single weight, one step at a time. | 24 min |
| **[Stochastic Gradient Descent, Clearly Explained!!!](https://www.youtube.com/watch?v=vMh0zPT0tLI)** — StatQuest | The minibatch idea of section 5. | 11 min |
| **[Gradient descent, how neural networks learn](https://www.youtube.com/watch?v=IHZwWFHWa-w)** — 3Blue1Brown | Gradient descent on a neural network's loss, as a preview of Meeting 17. | 21 min |
| **[Linear regression: Gradient descent](https://developers.google.com/machine-learning/crash-course/linear-regression/gradient-descent)** — Google Machine Learning Crash Course | Learning rate and loss curves, with an interactive exercise. | 15 min |

### Going further

| Resource | Why this one | Time |
|---|---|---|
| **[Why Momentum Really Works](https://distill.pub/2017/momentum/)** — Goh, *Distill*, 2017 | The eigenvalue analysis of sections 3 and 4, interactive, extended to momentum. | 45 min |
| **[An overview of gradient descent optimization algorithms](https://arxiv.org/abs/1609.04747)** — Ruder, 2016 | SGD, momentum, Adam and their relatives, side by side. | 45 min |
| **[Deep Learning, chapter 8: Optimization for Training Deep Models](https://www.deeplearningbook.org/contents/optimization.html)** — Goodfellow, Bengio, Courville | Non-convexity, saddle points, schedules, and early stopping as regularisation (section 7.8). | 2 h |
| **[Optimization Methods for Large-Scale Machine Learning](https://doi.org/10.1137/16M1080173)** — Bottou, Curtis, Nocedal, *SIAM Review*, 2018 | The theory of SGD: why noisy steps work, and convergence rates. | reference |